# 第 1 堂：語言模型在做什麼？（Colab 實作）

這本 notebook 是第 1 堂課的全部動手部分——不需要在自己電腦裝任何東西。

使用方式：`Runtime > Run all`，或一格一格跑。不需要 GPU。

> 先在下一格填入助教公布的課程 repo URL。

In [ ]:
REPO_URL = ""  # ← 填入助教公布的 repo URL，例如 https://github.com/<lab>/tiny_llm_edge_deploied.git

In [ ]:
import os
if not os.path.exists('course'):
    assert REPO_URL, '請先在上一格填入 REPO_URL'
    !git clone -q {REPO_URL} course
%cd course
!bash tools/build.sh

In [ ]:
%%bash
# 下載 baseline 模型（Karpathy 預訓練的 stories260K，26 萬參數）
mkdir -p models
base=https://huggingface.co/karpathy/tinyllamas/resolve/main/stories260K
for f in stories260K.bin stories260K.pt; do
  [ -f models/$f ] || wget -q $base/$f -O models/$f
done
ls -l models

## 示範 1：生成一個故事

這是一個 **26 萬參數、約 1MB** 的語言模型（對照：GPT-4 是兆級參數）。
它只在 TinyStories（給小小孩的英文短故事）上訓練過。

In [ ]:
!./bin/run models/stories260K.bin -z models/tok512.bin -t 0.8 -n 200 -i "Once upon a time"

## 示範 2：temperature 在控制什麼？

下面兩格：先用 `-t 0` 跑**兩次**，再用 `-t 1.4` 跑一次。

觀察並回答：`-t 0` 的兩次輸出有什麼關係？`-t 1.4` 發生了什麼？為什麼？

In [ ]:
!./bin/run models/stories260K.bin -z models/tok512.bin -t 0 -n 100 -i "Once upon a time"
print('=' * 60)
!./bin/run models/stories260K.bin -z models/tok512.bin -t 0 -n 100 -i "Once upon a time"

In [ ]:
!./bin/run models/stories260K.bin -z models/tok512.bin -t 1.4 -n 100 -i "Once upon a time"

## 示範 3：這個模型有多「懂」英文故事？

評分指標是 **bits-per-byte（BPB）**：模型平均要花多少 bits 才能「預測對」驗證集裡的每一個 byte。越低越好。

驗證集 `eval/validation_100.txt` 是 100 篇凍結的故事——之後品質榜就用它計分。

In [ ]:
!./bin/eval_bpb_f32 models/stories260K.bin -z models/tok512.bin -f eval/validation_100.txt -w 128

## 作業 1-3：window 實驗

把 `-w`（模型評分時能看到的 context 長度）換成 **64** 和 **256** 各跑一次，
記錄三個 BPB，並解釋變化的方向與原因。

In [ ]:
# 自己改 -w 的值
!./bin/eval_bpb_f32 models/stories260K.bin -z models/tok512.bin -f eval/validation_100.txt -w 64

## 作業（交一頁 markdown，詳見 `docs/handouts/lecture1.md`）

1. 貼一段你最喜歡的生成結果。
2. 重現 baseline 的 BPB = 0.8132（w=128）。
3. 上面的 window 實驗：三個數字 + 解釋。
4. 思考題：這個模型能把英文故事「壓縮」到每 byte 0.81 bits——這句話為什麼成立？跟 zip 有什麼關係？